# ITOM 6219 — Week 5: A/B Testing with Python

This notebook accompanies the Week 5 lecture notes on User Experience and A/B Testing. You will practice:

1. **Data Collection** — Retrieve experiment data from the Google Analytics API.
2. **Power Analysis** — Recommend an appropriate sample size before running an experiment.
3. **Hypothesis Testing** — Conduct Student's and Welch's t-tests to evaluate your hypothesis.
4. **Interpretation** — Compare the performance and assumptions of the two t-tests.

---
*© Jane Tan 2026. All rights reserved.*

---
# 1. Setup — Import Packages

Other than the statistical package `pingouin`, we introduce the **Google Analytics Data API**. Google has developed the `google.analytics.data` package to allow easy retrieval of historical reports.

Documentation: [Google Analytics Data API v1](https://developers.google.com/analytics/devguides/reporting/data/v1/rest)

In [ ]:
# ── 1.1 INSTALL PACKAGES ──────────────────────────────────────────────────
!pip3 install google.analytics.data
!pip3 install pingouin

Import the packages needed.

In [ ]:
# ── 1.2 IMPORT PACKAGES ───────────────────────────────────────────────────
from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.analytics.data_v1beta.types import (
    DateRange,
    Dimension,
    Metric,
    RunReportRequest,
    Filter,
    FilterExpression,
)
import os
import pandas as pd
import json
import scipy.stats as st
import numpy as np

---
# 2. An End-to-End A/B Test Example

Following the class notes, we will implement hypothesis testing for an experiment comparing the original website (control) to a variant in which the header bar color was changed (treatment).

## 2.1 Power Analysis — Sample Size

Before running an A/B test, it is important to conduct a **power analysis** to estimate the required sample size. This ensures your experiment is capable of detecting a meaningful effect if one exists. The sample size depends on the following key inputs:

- **Baseline performance** (e.g., current conversion rate)
- **Expected variant performance** (e.g., projected improvement)
- **Desired power** (typically 0.8 or 80%)
- **Significance level (alpha)** (commonly set at 0.05)

When launching a new experiment, you often have limited information about how the variant will perform, so you will need to make assumptions or use industry benchmarks. However, in a follow-up study — such as refining a previously tested feature — you can use prior results to make a more informed estimate, leading to a more accurate and efficient sample size calculation.

In [ ]:
# ── 2.1 POWER ANALYSIS ────────────────────────────────────────────────────
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# Inputs
baseline_rate = 0.5
variant_rate = 0.7
effect_size = proportion_effectsize(baseline_rate, variant_rate)

# Power analysis
analysis = NormalIndPower()
sample_size_per_group = analysis.solve_power(
    effect_size=effect_size, power=0.8, alpha=0.05, alternative='two-sided'
)

print(f"Required sample size per group: {int(sample_size_per_group)}")

## 2.2 Python Functions — A Quick Review

Before we build our data pipeline, let us review **user-defined functions** in Python. We discussed functions in Week 3 — they are prewritten blocks of code that can be invoked to carry out a specific set of actions. Below are three examples covering the basics.

**Example 1** — A function with no input and no return:

In [ ]:
# ── EXAMPLE 1: No input, no return ────────────────────────────────────────
def say_hello():
    print("Hello, class!")

say_hello()

**Example 2** — A function with one input:

In [ ]:
# ── EXAMPLE 2: One input ──────────────────────────────────────────────────
def get_dimensions(dimension):
    print(f"This is a report based on {dimension}!")

get_dimensions("device")
get_dimensions("date")

**Example 3** — A function with a return value:

In [ ]:
# ── EXAMPLE 3: With return ────────────────────────────────────────────────
def get_metrics(dimension):
    if dimension == "mobile":
        return 234
    else:
        return 0

result = get_metrics("mobile")
print(result)

---
## 2.3 Data Collection Pipeline

Now that we have reviewed functions, we will build a data pipeline to retrieve experiment data from Google Analytics and prepare it for hypothesis testing. The pipeline follows five steps:

1. **Step 1** — Get conversion sessions (sessions with a submit event) at a date level.
2. **Step 2** — Get total sessions at a date level.
3. **Step 3** — Perform a left join to merge the two datasets and calculate the submission ratio.
4. **Step 4** — Extract the ratio for treatment and control conditions.
5. **Step 5** — Run the t-test and interpret the results.

### 2.3.1 Conversion Report

Google Analytics provides APIs to retrieve data flexibly. Report generation is based on the definition of **dimensions** and **metrics**.

> More information about dimensions and metrics can be found [here](https://developers.google.com/analytics/devguides/reporting/data/v1/api-schema). More information about how to generate reports can be found [here](https://developers.google.com/analytics/devguides/reporting/data/v1/basics#python_3).

In [ ]:
# ── 2.3.1 CONVERSION REPORT ───────────────────────────────────────────────
# This function retrieves conversion sessions (submit events) from Google Analytics.
def sample_run_report_conversion(property_id="424145747"):
    """Runs a simple report on a Google Analytics 4 property."""
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'bigquery-419120-14d56b3f70eb.json'
    client = BetaAnalyticsDataClient()
    request = RunReportRequest(
        property="properties/{}".format(property_id),
        dimensions=[
            Dimension(name="campaignName"),
            Dimension(name="eventName"),
            Dimension(name="date"),
        ],
        metrics=[Metric(name="sessions")],
        date_ranges=[DateRange(start_date="2025-01-01", end_date="2025-04-07")],
        dimension_filter=FilterExpression(
            filter=Filter(
                field_name="campaignName",
                in_list_filter=Filter.InListFilter(
                    values=["main", "experiment"]
                ),
            )
        ),
    )
    response = client.run_report(request)
    return response

### 2.3.2 Session Report

The report below retrieves all experimental sessions (total number of sessions per day for each branch).

In [ ]:
# ── 2.3.2 SESSION REPORT ──────────────────────────────────────────────────
# This function retrieves total session counts from Google Analytics.
def sample_run_report_session(property_id="424145747"):
    """Runs a simple report on a Google Analytics 4 property."""
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'bigquery-419120-14d56b3f70eb.json'
    client = BetaAnalyticsDataClient()
    request = RunReportRequest(
        property="properties/{}".format(property_id),
        dimensions=[
            Dimension(name="customEvent:Branch"),
            Dimension(name="date"),
        ],
        metrics=[Metric(name="sessions")],
        date_ranges=[DateRange(start_date="2025-01-01", end_date="2025-04-07")],
        dimension_filter=FilterExpression(
            filter=Filter(
                field_name="customEvent:Branch",
                in_list_filter=Filter.InListFilter(
                    values=["main", "experiment"]
                ),
            )
        ),
    )
    response = client.run_report(request)
    return response

### 2.3.3 Data Conversion

The default API output is JSON. We transform it into a DataFrame using the helper function `response_to_df`.

In [ ]:
# ── 2.3.3 DATA CONVERSION ─────────────────────────────────────────────────
# Convert the Google Analytics API response into a pandas DataFrame.
def response_to_df(response):
    columns = []
    rows = []

    for col in response.dimension_headers:
        columns.append(col.name)
    for col in response.metric_headers:
        columns.append(col.name)

    for row_data in response.rows:
        row = []
        for val in row_data.dimension_values:
            row.append(val.value)
        for val in row_data.metric_values:
            row.append(val.value)
        rows.append(row)

    return pd.DataFrame(rows, columns=columns)

### 2.3.4 Step 1 — Conversion Data

Retrieve conversion sessions and filter for submit events only.

In [ ]:
# ── STEP 1: CONVERSION DATA ───────────────────────────────────────────────
response = sample_run_report_conversion(property_id="424145747")
df1 = response_to_df(response)

# Rename column to match the session report
df1.rename(columns={'campaignName': 'customEvent:Branch'}, inplace=True)

# Only look at submit events
df1 = df1[df1['eventName'] == 'submit']
df1

### 2.3.5 Step 2 — Session Data

Retrieve the total number of sessions per day for control and treatment.

In [ ]:
# ── STEP 2: SESSION DATA ──────────────────────────────────────────────────
response = sample_run_report_session(property_id="424145747")
df2 = response_to_df(response)
print(df2)

---
## 2.4 Hypothesis Testing

### 2.4.1 Step 3 — Merge Data

To combine the two datasets, we use the `merge()` function with `how='left'` — a **left join**. A left join keeps **all rows from the left table** (df2 — total sessions) and attaches matching rows from the right table (df1 — submit sessions). If a row in df2 has no match in df1, the joined columns are filled with `NaN`.

**df2 — Total Sessions (left table)**

| customEvent:Branch | date | sessions |
|---|---|---|
| main | 20250320 | 8 |
| experiment | 20250320 | 3 |
| main | 20250321 | 5 |

**df1 — Submit Sessions (right table)**

| customEvent:Branch | eventName | date | sessions |
|---|---|---|---|
| main | submit | 20250320 | 5 |
| main | submit | 20250321 | 2 |

**Result after left join** on `date` + `customEvent:Branch`:

| customEvent:Branch | date | sessions_x | eventName | sessions_y |
|---|---|---|---|---|
| main | 20250320 | 8 | submit | 5 |
| experiment | 20250320 | 3 | *NaN* | *NaN* |
| main | 20250321 | 5 | submit | 2 |

> The experiment branch on 20250320 had 3 total sessions but **no submit events** — the left join keeps this row and fills `sessions_y` with `NaN` (which we then replace with 0). This is why a left join is the right choice: we need every day to appear in the final data, even days with zero submissions.

In [ ]:
# ── STEP 3: MERGE DATA ────────────────────────────────────────────────────
# A left join keeps all rows from df2 and adds matching rows from df1.
# fillna('0') fills missing values for days with no submissions.
merged_df = pd.merge(df2, df1, on=['date', 'customEvent:Branch'], how='left')
merged_df['sessions_y'] = merged_df['sessions_y'].fillna('0')

### 2.4.2 Step 4 — Calculate the Submission Ratio

In [ ]:
# ── STEP 4: CALCULATE THE RATIO ───────────────────────────────────────────
# Calculate the ratio between submission sessions and all sessions per day.
merged_df['submit_session_ratio_per_day'] = (
    merged_df['sessions_y'].astype(int) / merged_df['sessions_x'].astype(int)
)

# Extract data for control (main branch)
control = merged_df[merged_df['customEvent:Branch'] == 'main']['submit_session_ratio_per_day']

# Extract data for treatment (experiment branch)
treatment = merged_df[merged_df['customEvent:Branch'] == 'experiment']['submit_session_ratio_per_day']

### 2.4.3 Step 5 — Hypothesis Testing

An independent t-test compares the means of two groups. The goal is to determine whether the treatment and control data are drawn from populations with the same mean (the null hypothesis) or different means (the alternative hypothesis). When we say "independent" samples, we mean there is no special relationship between observations in the two groups — a user is assigned to either treatment or control, not both.

The independent samples t-test comes in two forms: **Student's** and **Welch's**.

#### Student's T-Test (Equal Variance)

This test assumes that both populations have **identical variances**. It tests the null hypothesis that the two independent samples have identical average (expected) values.

In [ ]:
# ── STUDENT'S T-TEST (EQUAL VARIANCE) ─────────────────────────────────────
from pingouin import ttest

ttest(treatment, control, correction=False)

**More comprehensive results** — including mean differences and confidence intervals:

In [ ]:
# ── STUDENT'S T-TEST — DETAILED RESULTS ───────────────────────────────────
import pingouin as pg
import matplotlib.pyplot as plt

results = []

# Means
mean_treatment = treatment.mean()
mean_control = control.mean()

# Absolute and percentage difference
abs_diff = mean_treatment - mean_control
pct_diff = (abs_diff / mean_control) * 100

# T-test using pingouin
ttest = pg.ttest(treatment, control, correction=False)
ci_low, ci_high = ttest['CI95'].values[0]

results.append({
    'Test': "Student's T-Test",
    'Mean Treatment': mean_treatment,
    'Mean Control': mean_control,
    'Abs Diff': abs_diff,
    'Pct Diff (%)': pct_diff,
    'p-value': ttest['p_val'].values[0],
    'CI Lower': ci_low,
    'CI Upper': ci_high,
})

#### Welch's T-Test (Unequal Variance)

The biggest limitation of Student's t-test is the equal-variance assumption. This is rarely true in practice: if two samples do not have the same means, why should we expect them to have the same standard deviation? **Welch's t-test** relaxes this assumption, making it more conservative and more appropriate in most real-world scenarios.

In [ ]:
# ── WELCH'S T-TEST (UNEQUAL VARIANCE) ─────────────────────────────────────
from pingouin import ttest

ttest(treatment, control, correction=True)

**More comprehensive results:**

In [ ]:
# ── WELCH'S T-TEST — DETAILED RESULTS ─────────────────────────────────────
mean_treatment = treatment.mean()
mean_control = control.mean()

# Absolute and percentage difference
abs_diff = mean_treatment - mean_control
pct_diff = (abs_diff / mean_control) * 100

# T-test using pingouin (correction=True enables Welch's correction)
ttest = pg.ttest(treatment, control, correction=True)
ci_low, ci_high = ttest['CI95'].values[0]

results.append({
    'Test': "Welch's T-Test",
    'Mean Treatment': mean_treatment,
    'Mean Control': mean_control,
    'Abs Diff': abs_diff,
    'Pct Diff (%)': pct_diff,
    'p-value': ttest['p_val'].values[0],
    'CI Lower': ci_low,
    'CI Upper': ci_high,
})

#### Comparison of Results

In [ ]:
# ── COMPARISON TABLE ──────────────────────────────────────────────────────
results_df = pd.DataFrame(results)
results_df

#### [Optional] Visualize the Results

In [ ]:
# ── VISUALIZATION ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))

ax.errorbar(
    x=results_df['Abs Diff'],
    y=results_df['Test'],
    xerr=[
        results_df['Abs Diff'] - results_df['CI Lower'],
        results_df['CI Upper'] - results_df['Abs Diff'],
    ],
    fmt='o',
    capsize=5,
    linestyle='none',
)

ax.axvline(0, color='gray', linestyle='--')
ax.set_xlabel("Difference in Mean Sessions")
ax.set_title("Difference in Sessions (with 95% CI)")
plt.tight_layout()
plt.show()